# Gemma 4 × GitHub Agent (Step 18)

Wire a local **Gemma 4** model to the **GitHub MCP server** for intelligent repo
management: issue triage, PR summarisation, code search, and automated responses —
all running privately on your machine.

## What you need
- [Ollama](https://ollama.com) running with `gemma4:12b` pulled
- `llama-index-llms-ollama`, `llama-index-tools-mcp` installed
- `GITHUB_PERSONAL_ACCESS_TOKEN` with `repo` and `read:org` scopes
- Node.js ≥18

```bash
pip install llama-index-llms-ollama llama-index-tools-mcp
ollama pull gemma4:12b
export GITHUB_PERSONAL_ACCESS_TOKEN=ghp_...
```

## Part 1 — Connect Gemma 4 to the GitHub MCP Server

In [ ]:
import os
import asyncio

from llama_index.llms.ollama import Ollama
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.core.agent.workflow import ReActAgent

GITHUB_TOKEN = os.environ.get("GITHUB_PERSONAL_ACCESS_TOKEN", "")
REPO_OWNER = os.environ.get("GITHUB_REPO_OWNER", "morongosteve")
REPO_NAME = os.environ.get("GITHUB_REPO_NAME", "llama_index")

if not GITHUB_TOKEN:
    print("Set GITHUB_PERSONAL_ACCESS_TOKEN to enable live GitHub tools.")

llm = Ollama(model="gemma4:12b", request_timeout=120.0)

In [ ]:
github_client = BasicMCPClient(
    "npx",
    args=["-y", "@modelcontextprotocol/server-github"],
    env={**os.environ, "GITHUB_PERSONAL_ACCESS_TOKEN": GITHUB_TOKEN},
)

github_spec = McpToolSpec(client=github_client)
github_tools = await github_spec.to_tool_list_async()

print(f"Loaded {len(github_tools)} GitHub tools:")
for t in github_tools:
    print(f"  • {t.metadata.name}")

In [ ]:
github_agent = ReActAgent(
    tools=github_tools,
    llm=llm,
    max_iterations=15,
    verbose=True,
)

## Part 2 — Issue Triage

Gemma reads open issues and categorises them by type, priority, and effort.

In [ ]:
response = await github_agent.run(
    f"List all open issues in {REPO_OWNER}/{REPO_NAME}. "
    "For each issue, classify it as: bug / feature request / docs / question / other. "
    "Also estimate priority (high/medium/low) based on the title and body. "
    "Present a summary table."
)
print(response)

In [ ]:
# Gemma suggests labels for a specific issue
ISSUE_NUMBER = 1  # change to your issue number
response = await github_agent.run(
    f"Read issue #{ISSUE_NUMBER} in {REPO_OWNER}/{REPO_NAME}. "
    "Summarise it in 2 sentences, suggest 3 relevant labels, and draft a helpful first response "
    "that acknowledges the issue and asks any clarifying questions needed."
)
print(response)

## Part 3 — Pull Request Summaries

In [ ]:
response = await github_agent.run(
    f"List the 5 most recent pull requests in {REPO_OWNER}/{REPO_NAME} (open and merged). "
    "For each PR: title, author, status, number of files changed, and a one-sentence summary "
    "of what the change does."
)
print(response)

In [ ]:
# Deep-dive on a specific PR — Gemma writes a review summary
PR_NUMBER = 75  # change to your PR
response = await github_agent.run(
    f"Get the diff for PR #{PR_NUMBER} in {REPO_OWNER}/{REPO_NAME}. "
    "Summarise: (1) what changed and why, (2) any potential issues or missing tests, "
    "(3) whether the PR description matches the actual diff. "
    "Keep the summary under 200 words."
)
print(response)

## Part 4 — Code Search & Q&A

In [ ]:
# Search for a pattern across the repo
response = await github_agent.run(
    f"Search the code in {REPO_OWNER}/{REPO_NAME} for any usage of 'ImageBlock'. "
    "List the files and line numbers where it appears, and summarise the usage patterns."
)
print(response)

In [ ]:
# Ask a natural-language code question
response = await github_agent.run(
    f"In the {REPO_OWNER}/{REPO_NAME} repo, how does the Ollama LLM integration "
    "handle multimodal image inputs? Read the relevant source files and explain "
    "the data flow from user input to the Ollama API call."
)
print(response)

## Part 5 — Automated Changelog Generation

In [ ]:
response = await github_agent.run(
    f"List all commits merged to main in {REPO_OWNER}/{REPO_NAME} in the last 7 days. "
    "Group them by category (feat / fix / docs / chore) based on the commit message prefix. "
    "Generate a clean CHANGELOG.md entry in Keep-a-Changelog format."
)
print(response)

## Part 6 — Combined: Code RAG + GitHub Agent

Index the repo locally for fast semantic search, then use the GitHub agent for
live data (issues, PRs, commits).

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.tools import QueryEngineTool
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Fully local embeddings — nothing sent to the cloud
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = llm

# Index the Ollama integration source
code_docs = SimpleDirectoryReader(
    "../../llama-index-integrations/llms/llama-index-llms-ollama/llama_index/llms/ollama",
    recursive=True,
).load_data()
code_index = VectorStoreIndex.from_documents(code_docs)
code_engine = code_index.as_query_engine(similarity_top_k=5)

code_tool = QueryEngineTool.from_defaults(
    query_engine=code_engine,
    name="ollama_source_search",
    description="Search the local Ollama LLM integration source code for implementation details.",
)

hybrid_agent = ReActAgent(
    tools=[code_tool] + github_tools,
    llm=llm,
    max_iterations=15,
    verbose=True,
)

print("Hybrid agent ready: local code RAG + live GitHub tools")

In [ ]:
# Cross-reference: does the latest PR match what the source says it should do?
response = await hybrid_agent.run(
    f"Using both the local source code and the latest PRs in {REPO_OWNER}/{REPO_NAME}, "
    "explain how multimodal image support works in the Ollama integration. "
    "Cross-reference the source with PR #68 to confirm the implementation matches the description."
)
print(response)

## Part 7 — Human-in-the-Loop Write Actions

Gemma proposes; you approve before any GitHub write (comment, label, close).

In [ ]:
def confirm_github_action(agent, prompt: str, description: str) -> str:
    """Human-in-the-loop wrapper for GitHub write actions."""
    answer = input(f"\n[CONFIRM] {description}\nProceed? (yes/no): ").strip().lower()
    if answer != "yes":
        return "Action cancelled."
    import asyncio
    return asyncio.get_event_loop().run_until_complete(agent.run(prompt))

In [ ]:
# Draft a response to an issue, then post with confirmation
ISSUE_TO_RESPOND = 1  # set to the issue number

# First: Gemma drafts (read only, no confirmation needed)
draft = await github_agent.run(
    f"Read issue #{ISSUE_TO_RESPOND} in {REPO_OWNER}/{REPO_NAME} "
    "and draft a helpful, friendly response. Don't post it yet."
)
print("Gemma's draft response:")
print(draft)

# Then: post with confirmation
confirm_github_action(
    github_agent,
    f"Post the following comment on issue #{ISSUE_TO_RESPOND} in {REPO_OWNER}/{REPO_NAME}:\n{draft}",
    f"Post a comment on issue #{ISSUE_TO_RESPOND}",
)

## Security reminders

- `GITHUB_PERSONAL_ACCESS_TOKEN` — store in `.env`, never commit
- Use a **fine-grained** token scoped to your specific repos with minimum permissions
- For read-only work (code search, issue/PR summaries): use `Contents: read` + `Issues: read` scopes only
- For write actions (comments, labels): add `Issues: write` — keep `Administration` and `Secrets` always off
- All write actions go through `confirm_github_action` — the agent never auto-posts